In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_pickle("../02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl")
print("shape:", df.shape)
display(df.dtypes.to_frame("dtype"))


shape: (6365, 21)


,dtype
id,int64
origen,object
fuente,object
no_enviar_email,object
no_llamar,object
compra,int64
visitas_total,float64
tiempo_en_site_total,int64
paginas_vistas_visita,float64
ult_actividad,object


##  Análisis estructural


In [32]:
import re
import unicodedata

def normalizar_nombre(s):
    s = unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode('ascii')
    s = re.sub(r'[^a-zA-Z0-9]+', '_', s).strip('_').lower()
    return s
nombres = pd.DataFrame({'original': df.columns, 'normalizado': [normalizar_nombre(c) for c in df.columns]})
nombres['cambio'] = nombres['original'] != nombres['normalizado']
print('Nombres con cambios:', int(nombres['cambio'].sum()))
print('Colisiones tras normalizar:', int(nombres['normalizado'].duplicated().sum()))
##display(nombres)


Nombres con cambios: 0
Colisiones tras normalizar: 0


In [33]:
filas=[]
for c in df.columns:
    serie=df[c]
    muestra=serie.dropna().head(5).tolist()
    incompatible='—'
    sugerido=str(serie.dtype)
    justificacion='Tipo actual coherente con los valores observados.'
    if serie.dtype == 'object':
        numericos=pd.to_numeric(serie, errors='coerce')
        pct_num=float(numericos.notna().mean()*100)
        unicos=serie.dropna().astype(str).str.strip().str.lower().unique()
        if pct_num >= 95 and len(unicos)>0:
            sugerido='numérico'; justificacion='Al menos 95% de los valores son convertibles a número.'
        elif set(unicos).issubset({'si','no','yes','true','false','0','1'}):
            sugerido='booleano/categórico'; justificacion='Valores compatibles con una variable binaria.'
        else:
            sugerido='categórico'; justificacion='Valores textuales o categorías; conservar como texto.'
        incompatible=f'{100-pct_num:.2f}% convertibles a número'
    filas.append({'columna':c,'tipo_actual':str(serie.dtype),'tipo_sugerido':sugerido,'nulos':int(serie.isna().sum()),'muestra':str(muestra),'incompatibilidad':incompatible,'justificación':justificacion})
tipos=pd.DataFrame(filas)
display(tipos)


,columna,tipo_actual,tipo_sugerido,nulos,muestra,incompatibilidad,justificación
0,id,int64,int64,0,"[630952, 633132, 636677, 601564, 645378]",—,Tipo actual coherente con los valores observados.
1,origen,object,categórico,0,"['Landing Page Submission', 'API', 'Landing Pa...",100.00% convertibles a número,Valores textuales o categorías; conservar como...
2,fuente,object,categórico,25,"['Google', 'Chat', 'Google', 'Chat', 'Google']",100.00% convertibles a número,Valores textuales o categorías; conservar como...
3,no_enviar_email,object,booleano/categórico,0,"['No', 'No', 'No', 'No', 'No']",100.00% convertibles a número,Valores compatibles con una variable binaria.
4,no_llamar,object,booleano/categórico,0,"['No', 'No', 'No', 'No', 'No']",100.00% convertibles a número,Valores compatibles con una variable binaria.
5,compra,int64,int64,0,"[0, 0, 1, 0, 0]",—,Tipo actual coherente con los valores observados.
6,visitas_total,float64,float64,92,"[4.0, 0.0, 4.0, 0.0, 3.0]",—,Tipo actual coherente con los valores observados.
7,tiempo_en_site_total,int64,int64,0,"[1221, 0, 963, 0, 172]",—,Tipo actual coherente con los valores observados.
8,paginas_vistas_visita,float64,float64,92,"[4.0, 0.0, 4.0, 0.0, 3.0]",—,Tipo actual coherente con los valores observados.
9,ult_actividad,object,categórico,68,"['Email Opened', 'Chat Conversation', 'SMS Sen...",100.00% convertibles a número,Valores textuales o categorías; conservar como...


In [34]:
print('Duplicados de filas:', int(df.duplicated().sum()))
print('Duplicados de id:', int(df['id'].duplicated().sum()))
print('id único:', bool(df['id'].is_unique))


Duplicados de filas: 0
Duplicados de id: 0
id único: True


##  Análisis de contenido


Las columnas de score_actividad y score-perfil nos llegan a través de la herramienta de marketing automation, mostrando la actividad previa de ese usuario. si el usuario es nuevo, es lógico que no tenga datos. Por eso tiene un porcentaje tan alto de missings. No las eliminamos

Los missings de ocupación, ámbito, visitas total, paginas_vistas_visita, ult_actividad pueden imputarse. agregamos indicadores como visitas_total_missing para que el modelo conozca que el valor fue imputado

In [35]:
missing = pd.DataFrame({'columna': df.columns, 'nulos': df.isna().sum().values, 'porcentaje': (df.isna().mean()*100).round(2).values})
missing = missing[missing['nulos'] > 0].sort_values('porcentaje', ascending=False)
print('Columnas con ausentes:', len(missing))
display(missing)


Columnas con ausentes: 8


,columna,nulos,porcentaje
18,score_actividad,2927,45.99
19,score_perfil,2927,45.99
11,ocupacion,1888,29.66
10,ambito,1012,15.90
8,paginas_vistas_visita,92,1.45
6,visitas_total,92,1.45
9,ult_actividad,68,1.07
2,fuente,25,0.39


In [36]:
cat_rows=[]
for c in df.select_dtypes(include='object').columns:
    s=df[c].fillna('<NA>').astype(str).str.strip()
    vc=s.value_counts()
    cat_rows.append({'columna':c,'cardinalidad':int(s.nunique()),'moda':str(vc.index[0]),'frecuencia_moda':int(vc.iloc[0]),'etiquetas_1pct':int((vc/len(df) < .01).sum()),'valores_muestra':', '.join(map(str,vc.index[:8]))})
cat_summary=pd.DataFrame(cat_rows).sort_values('cardinalidad',ascending=False)
display(cat_summary)


,columna,cardinalidad,moda,frecuencia_moda,etiquetas_1pct,valores_muestra
5,ambito,20,Select,1263,3,"Select, <NA>, Finance Management, Marketing Ma..."
1,fuente,16,Google,2013,10,"Google, Direct Traffic, Chat, Organic Search, ..."
4,ult_actividad,16,Email Opened,2371,6,"Email Opened, SMS Sent, Chat Conversation, Pag..."
6,ocupacion,7,Unemployed,3799,3,"Unemployed, <NA>, Working Professional, Studen..."
0,origen,4,Landing Page Submission,3411,1,"Landing Page Submission, API, Lead Add Form, L..."
3,no_llamar,2,No,6363,1,"No, Yes"
2,no_enviar_email,2,No,5863,0,"No, Yes"
7,conociste_google,2,No,6358,1,"No, Yes"
12,conociste_referencias,2,No,6360,1,"No, Yes"
11,conociste_facebook,2,No,6363,1,"No, Yes"


In [37]:
num_rows=[]
for c in df.select_dtypes(include=np.number).columns:
    s=df[c].dropna()
    q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    low=q1-1.5*iqr; high=q3+1.5*iqr
    out=((s<low)|(s>high)).sum()
    num_rows.append({'columna':c,'media':round(s.mean(),2),'mediana':round(s.median(),2),'p01':round(s.quantile(.01),2),'p99':round(s.quantile(.99),2),'mínimo':round(s.min(),2),'máximo':round(s.max(),2),'outliers_iqr':int(out),'pct_outliers':round(out/len(s)*100,2)})
num_summary=pd.DataFrame(num_rows)
display(num_summary)


,columna,media,mediana,p01,p99,mínimo,máximo,outliers_iqr,pct_outliers
0,id,617039.78,615216.0,580304.72,659410.12,579533.0,660728.0,0,0.00
1,compra,0.37,0.0,0.00,1.00,0.0,1.0,0,0.00
2,visitas_total,3.47,3.0,0.00,16.00,0.0,251.0,179,2.85
3,tiempo_en_site_total,496.10,252.0,0.00,1846.36,0.0,2272.0,0,0.00
4,paginas_vistas_visita,2.39,2.0,0.00,9.00,0.0,24.0,241,3.84
5,score_actividad,14.32,14.0,10.00,17.00,7.0,18.0,499,14.51
6,score_perfil,16.36,16.0,13.00,20.00,11.0,20.0,0,0.00


37% de nuestros clientes nos termina comprando
Visitas_total tiene un valor maximo demasiado alto, ¿puede ser un bot?
tiempo_en_site_total: media 496 vs mediana 252.  Pocos usuarios pasan mucho tiempo.

In [38]:
print('Valores vacíos o marcadores ocultos:')
hidden=[]
for c in df.select_dtypes(include='object').columns:
    s=df[c].dropna().astype(str)
    count=int(s.str.strip().isin(['','-','N/A','NA','null','None']).sum())
    if count: hidden.append({'columna':c,'marcadores':count})
display(pd.DataFrame(hidden) if hidden else pd.DataFrame({'resultado':['No se detectaron marcadores ocultos']}))


Valores vacíos o marcadores ocultos:


,resultado
0,No se detectaron marcadores ocultos


## Correcciones
Se preservan los scores como `NaN`; se imputan las variables aprobadas y se agregan indicadores de faltantes.


In [39]:
transform_steps = []
for c in ["ocupacion", "ambito", "ult_actividad", "fuente"]:
    df[c] = df[c].fillna("Desconocido")
    transform_steps.append({"op":"impute", "column":c, "strategy":"constant", "value":"Desconocido"})
for c in ["visitas_total", "paginas_vistas_visita"]:
    missing_col = f"{c}_missing"
    df[missing_col] = df[c].isna().astype("int8")
    median = float(df[c].median())
    df[c] = df[c].fillna(median)
    transform_steps.append({"op":"impute", "column":c, "strategy":"median", "value":median})
    transform_steps.append({"op":"missing_indicator", "column":missing_col, "source_column":c})
print("shape_after_lote_b:", df.shape)
print("remaining_missing:")
display(df.isna().sum().loc[lambda s: s > 0].to_frame("nulos"))
print("new_columns:", ["visitas_total_missing", "paginas_vistas_visita_missing"])


shape_after_lote_b: (6365, 23)
remaining_missing:


,nulos
score_actividad,2927
score_perfil,2927


new_columns: ['visitas_total_missing', 'paginas_vistas_visita_missing']


Se unifica la categoría `google` con `Google`, preservando el resto de etiquetas.

In [40]:
fuente_map = {"google": "Google"}
df["fuente"] = df["fuente"].replace(fuente_map)
transform_steps.append({"op":"unify_categories", "column":"fuente", "mapping":fuente_map})
print("google_exact_count:", int((df["fuente"] == "Google").sum()))
print("lowercase_google_remaining:", int((df["fuente"] == "google").sum()))


google_exact_count: 2017
lowercase_google_remaining: 0


In [41]:
df_original = pd.read_pickle("../02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl")
def resumen_descendente(serie, nombre):
    s = serie.dropna()
    freq = s.value_counts().sort_index(ascending=False).rename_axis(nombre).reset_index(name="registros")
    freq["porcentaje"] = (freq["registros"] / len(s) * 100).round(3)
    freq["porcentaje_acumulado_desde_arriba"] = freq["porcentaje"].cumsum().round(3)
    print(f"{nombre}: n={len(s)}, valores_unicos={s.nunique()}, mínimo={s.min()}, máximo={s.max()}")
    print("\nFrecuencias desde el máximo:")
    display(freq.head(30))
    print("\nFrecuencias alrededor de la zona baja:")
    display(freq.tail(20))
    return freq
freq_visitas = resumen_descendente(df_original["visitas_total"], "visitas_total")
freq_paginas = resumen_descendente(df_original["paginas_vistas_visita"], "paginas_vistas_visita")


visitas_total: n=6273, valores_unicos=35, mínimo=0.0, máximo=251.0

Frecuencias desde el máximo:


,visitas_total,registros,porcentaje,porcentaje_acumulado_desde_arriba
0,251.0,1,0.016,0.016
1,141.0,1,0.016,0.032
2,54.0,1,0.016,0.048
3,41.0,1,0.016,0.064
4,30.0,1,0.016,0.080
5,29.0,2,0.032,0.112
6,28.0,1,0.016,0.128
7,27.0,2,0.032,0.160
8,26.0,2,0.032,0.192
9,25.0,4,0.064,0.256



Frecuencias alrededor de la zona baja:


,visitas_total,registros,porcentaje,porcentaje_acumulado_desde_arriba
15,19.0,5,0.080,0.687
16,18.0,8,0.128,0.815
17,17.0,11,0.175,0.990
18,16.0,18,0.287,1.277
19,15.0,11,0.175,1.452
20,14.0,22,0.351,1.803
21,13.0,34,0.542,2.345
22,12.0,32,0.510,2.855
23,11.0,53,0.845,3.700
24,10.0,88,1.403,5.103



Percentiles:


...

,paginas_vistas_visita,registros,porcentaje,porcentaje_acumulado_desde_arriba
0,24.00,1,0.016,0.016
1,16.00,2,0.032,0.048
2,15.00,1,0.016,0.064
3,14.50,1,0.016,0.080
4,14.00,6,0.096,0.176
5,13.00,5,0.080,0.256
6,12.00,5,0.080,0.336
7,11.50,1,0.016,0.352
8,11.00,8,0.128,0.480
9,10.00,20,0.319,0.799



Frecuencias alrededor de la zona baja:


,paginas_vistas_visita,registros,porcentaje,porcentaje_acumulado_desde_arriba
75,1.57,3,0.048,65.352
76,1.56,2,0.032,65.384
77,1.54,1,0.016,65.400
78,1.50,210,3.348,68.748
79,1.48,1,0.016,68.764
80,1.45,1,0.016,68.780
81,1.43,3,0.048,68.828
82,1.40,6,0.096,68.924
83,1.38,3,0.048,68.972
84,1.33,48,0.765,69.737



Percentiles:


,percentil,valor
0,p50,2.00
1,p75,3.33
2,p90,5.00
3,p95,6.00
4,p97.5,7.00
5,p99,9.00
6,p99.5,10.00
7,p99.9,14.00


In [ ]:
mask_extremos = (df["visitas_total"] > 30) | (df["paginas_vistas_visita"] > 20)
filas_eliminadas = int(mask_extremos.sum())
ids_eliminados = df.loc[mask_extremos, "id"].tolist()
df = df.loc[~mask_extremos].copy()
transform_steps.append({"op":"drop_row", "predicate":"visitas_total > 30 OR paginas_vistas_visita > 20", "rows_removed":filas_eliminadas, "ids_removed":ids_eliminados})
print("rows_removed:", filas_eliminadas)
print("ids_removed:", ids_eliminados)
print("shape_after_outlier_trim:", df.shape)

`id` se conserva para trazabilidad, pero debe excluirse como predictor durante el modelado. No se modifica el dataframe en esta fase.


In [11]:
modeling_exclusions = ["id"]
print("modeling_exclusions:", modeling_exclusions)
print("df_shape_before_finalization:", df.shape)


modeling_exclusions: ['id']
df_shape_before_finalization: (6365, 23)


## Finalización de calidad
Persistencia del dataframe limpio, reporte, especificación de transformaciones y documentación del estado del proyecto.


In [ ]:
import io
import json
from datetime import date
from pathlib import Path

out_dir = Path("../06_resultados/Calidad_Datos")
out_dir.mkdir(parents=True, exist_ok=True)
clean_path = Path("../02_datos/03_Entrenamiento/02_train_tablon_calidad.pkl")
report_path = out_dir / "informe_calidad_datos.md"
spec_path = out_dir / "transformaciones.json"
df.to_pickle(clean_path)

buffer = io.StringIO()
df.info(buf=buffer)
info_text = buffer.getvalue()

report = f"""# Informe de calidad de datos

## Resumen

- Filas iniciales del train: 6365
- Filas finales: {len(df)}
- Columnas finales: {len(df.columns)}
- Fecha: {date.today().isoformat()}

## Problemas detectados y decisiones

- Los nombres de columnas ya estaban normalizados: 0 cambios y 0 colisiones.
- No se aplicó deduplicación: el análisis previo encontró 0 filas duplicadas e `id` único.
- `score_actividad` y `score_perfil` conservan sus faltantes estructurales para usuarios nuevos.
- Faltantes categóricos en `ocupacion`, `ambito`, `ult_actividad` y `fuente` se imputaron como `Desconocido`.
- Faltantes numéricos en `visitas_total` y `paginas_vistas_visita` se imputaron con la mediana del train y se agregaron indicadores de missingness.
- `fuente` se normalizó unificando `google` con `Google`.
- Se eliminaron registros con `visitas_total > 30` o `paginas_vistas_visita > 20`: 5 filas.
- `id` se conserva para trazabilidad, pero se excluye como predictor.

## Transformaciones por variable

| Variable | Transformaciones aplicadas |
|---|---|
| `ocupacion` | Imputación constante `Desconocido` |
| `ambito` | Imputación constante `Desconocido` |
| `ult_actividad` | Imputación constante `Desconocido` |
| `fuente` | Imputación constante `Desconocido`; `google` → `Google` |
| `visitas_total` | Imputación con mediana; eliminación de filas > 30; indicador `visitas_total_missing` |
| `paginas_vistas_visita` | Imputación con mediana; eliminación de filas > 20; indicador `paginas_vistas_visita_missing` |
| `score_actividad` | Sin transformación; se preservan NaN estructurales |
| `score_perfil` | Sin transformación; se preservan NaN estructurales |
| `id` | Sin transformación; excluir durante modelado |

## Estructura final

```
{info_text}```

## Artefactos

- Dataframe limpio: `02_datos/03_Entrenamiento/02_train_tablon_calidad.pkl`
- Especificación: `06_resultados/Calidad_Datos/transformaciones.json`
"""
report_path.write_text(report, encoding="utf-8")
spec = {"version": 1, "fitted_on": "02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl", "steps": transform_steps}
spec_path.write_text(json.dumps(spec, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

copilot_path = Path("../.github/copilot-instructions.md")
copilot = copilot_path.read_text(encoding="utf-8") if copilot_path.exists() else ""
section = f"""## ESTADO ACTUAL DEL PROYECTO

**Dataframe actual**: `../02_datos/03_Entrenamiento/02_train_tablon_calidad.pkl`

**Estructura del dataframe**:
```
{info_text}```
"""
marker = "## ESTADO ACTUAL DEL PROYECTO"
if marker in copilot: copilot = copilot[:copilot.index(marker)].rstrip() + "\n\n" + section
else: copilot = copilot.rstrip() + "\n\n" + section
copilot_path.parent.mkdir(parents=True, exist_ok=True)
copilot_path.write_text(copilot, encoding="utf-8")
print("clean_saved:", clean_path.exists())
print("report_saved:", report_path.exists())
print("spec_saved:", spec_path.exists())
print("copilot_updated:", copilot_path.exists())
print("final_shape:", df.shape)
